In [1]:
import numpy as np
import xgboost as xgb
import scipy.stats as stats
import scipy.special as special
import openturns as ot
from matplotlib import pyplot as plt
from typing import Dict, List, Tuple

In [2]:
import sys
import os

# Set project root relative to notebook location
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
from Losses import gaussian_xgb_loss as gauss
from Losses import clayton_xgb_loss as clayton
from Losses import gumbel_xgb_loss as gumbel

In [4]:
def sample_gaussian_copula(rho_array, seed):
    """
    Sample one bivariate observation from a Gaussian copula for each rho in rho_array.

    Parameters:
        rho_array (np.ndarray): shape (n,), correlation coefficients in (-1, 1)

    Returns:
        np.ndarray: shape (n, 2), samples from the Gaussian copula
    """
    ot.RandomGenerator.SetSeed(seed)
    n = len(rho_array)
    samples = np.zeros((n, 2))

    for i, rho in enumerate(rho_array):
        if not (-1 < rho < 1):
            raise ValueError("Each rho must be in the interval (-1, 1).")

        # Define correlation matrix
        corr_matrix = ot.CorrelationMatrix(2)
        corr_matrix[0, 1] = rho

        # Create the Gaussian copula
        copula = ot.NormalCopula(corr_matrix)

        # Draw a single sample
        u = copula.getSample(1)[0]
        samples[i] = [u[0], u[1]]

    samples = np.array(samples)
    # Clip to avoid 0 or 1
    eps = 1e-10
    samples = np.clip(samples, eps, 1 - eps)
    return samples

def sample_clayton_copula(theta_array, seed):
    """
    Sample one point from a bivariate Clayton copula for each parameter value in theta_array.
    
    Parameters:
        theta_array (np.ndarray): 1D array of positive Clayton copula parameters (theta > 0).
    
    Returns:
        samples (np.ndarray): Array of shape (n, 2) where each row is a sample (u1, u2).
    """
    ot.RandomGenerator.SetSeed(seed)
    samples = []
    for theta in theta_array:
        copula = ot.ClaytonCopula(theta)
        sample = copula.getSample(1)
        samples.append(sample[0])

    samples = np.array(samples)
    eps = 1e-10
    samples = np.clip(samples, eps, 1 - eps)
    return samples

def sample_gumbel_copula(theta_array, seed):
    """
    Sample one observation from a bivariate Gumbel copula for each theta.
    Uses OpenTURNS for sampling.
    
    Parameters:
        theta_array (np.ndarray): array of Gumbel copula parameters (theta >= 1)
    
    Returns:
        np.ndarray: shape (n, 2), copula samples in (0, 1)
    """
    ot.RandomGenerator.SetSeed(seed)
    samples = []
    for theta in theta_array:
        copula = ot.GumbelCopula(theta)
        sample = copula.getSample(1)
        samples.append(sample[0])
    
    samples = np.array(samples)
    # Clip to avoid 0 or 1
    eps = 1e-10
    samples = np.clip(samples, eps, 1 - eps)
    return samples

#def generate_copula_data(family, tau, size, ratio, seed) -> Tuple[xgb.DMatrix, xgb.DMatrix]:
    '''Generate data'''
    match family:
        case 'gaussian':
            y = sample_gaussian_copula(gauss.gauss_tau2par(tau), seed)
        case 'clayton':
            y = sample_clayton_copula(clayton.clayton_tau2par(tau), seed)
        case 'gumbel':
            y = sample_gumbel_copula(gumbel.gumbel_tau2par(tau), seed)
        case _:
            return raiseValueError("Could not match the copula family")

    train_portion = int(size * ratio)

    train_x: np.ndarray = x[: train_portion]
    train_y: np.ndarray = y[: train_portion]
    dtrain = xgb.DMatrix(train_x, label=train_y)

    test_x = x[train_portion:]
    test_y = y[train_portion:]
    dtest = xgb.DMatrix(test_x, label=test_y)
    return dtrain, dtest

In [17]:
# shape of generated data.
n = 6000
p = 10

ratio = 0.75
seed = 2000

num_boost_round = 500

np.random.seed(seed=seed)

x = np.random.uniform(0, 1, (n, p))
#x = np.random.randn(kRows, kCols)
x1 = x[:, 0]
x2 = x[:, 1]
x3 = x[:, 2]

#theta = np.exp(0.01 - 0.2*(x1 < 0.3) + 0.2*np.sin(2*np.pi*x2) + x3**2 - x3*(np.abs(x2) > 0.5))
# First linear scenario
tau1 = special.expit(0.5 - 0.3 * x1 + 0.4*x2 - 2*x3)
# Second non-linear scenario
tau2 = special.expit(0.1 - 0.2*(x1 < 0.3) + 0.3*np.sin(2*np.pi*x2) + 0.5*x3**2 - 2.5*x3*(np.abs(x2) > 0.5))
#print(np.min(tau1))
#print(np.max(tau1))
    
#depA = (x1 < 0.4) & (x2 < 0.75)
#depB = (x1 >= 0.4) & (x2 < 0.75)
#depC = (x1 < 0.4) & (x2 >= 0.75)
#depD = (x1 >= 0.4) & (x2 >= 0.75)

#tau = np.zeros(n)
#tau[depA] = 0.1
#tau[depB] = 0.3
#tau[depC] = 0.5
#tau[depD] = 0.7

In [6]:
def gaussian_lpdf(u, rho):
    """
    Gaussian copula log-likelihood evaluation

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Gaussian copula log-likelihood
    """
    a = stats.norm.ppf(u[:, 0]).reshape(-1,1)
    b = stats.norm.ppf(u[:, 1]).reshape(-1,1)
    c = a*b
    d = a**2 + b**2
    return np.sum(-np.log(np.sqrt(1-rho**2)) - ((rho**2)*d - 2*rho*c)/(2*(1-rho**2)))

def gaussian_lpdf_grad(u, rho):
    """
    Gaussian copula gradient w.r.t copula parameter rho

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Gaussian copula lpdf gradient
    """
    a = stats.norm.ppf(u[:, 0]).reshape(-1,1)
    b = stats.norm.ppf(u[:, 1]).reshape(-1,1)
    c = a*b
    d = a**2 + b**2
    return rho/(1-rho**2) + ((1+rho**2)*c - d*rho)/((1-rho**2)**2)
    
#def gaussian_link(par):
    """
    :param par: current parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: link function of copula parameter
    """
    return np.tan((np.pi/2)*par)
    
def gaussian_inv_link(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: inverse link function of transformed copula parameter
    """
    return (2/np.pi)*np.arctan(tpar)
    
def gaussian_inv_link_grad(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    return: gradient of inverse link function w.r.t transformed copula parameter
    """
    return (2/np.pi)*(1/(np.square(tpar)+1))
    
def gaussian_gradient(u: np.ndarray, predt: np.ndarray) -> np.ndarray:
    '''Compute the gradient of the nll'''
    #u = dtrain.get_label()
    #tau = gauss_par2tau(gaussian_inv_link(predt))
    #rho = gauss_tau2par(tau)
    rho = gaussian_inv_link(predt)
    return -gaussian_lpdf_grad(u, rho)*gaussian_inv_link_grad(predt)
        
def gaussian_copula_nll_obj(u: np.ndarray, predt: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    '''Gaussian copula negative log-likelihood objective'''
    grad = gaussian_gradient(u, predt)
    hess = grad**2 #BHHH hessian substitute
    return grad, hess

def gaussian_copula_nll_metric(u: np.ndarray, predt: np.ndarray) -> Tuple[str, float]:
    ''' Gaussian copula negative log-likelihood metric'''
    #u = dtrain.get_label()
    #tau = gauss_par2tau(gaussian_inv_link(predt))
    #rho = gauss_tau2par(tau)
    rho = gaussian_inv_link(predt)
    log_likelihood = gaussian_lpdf(u, rho)
    return float(-log_likelihood)

In [7]:
def clayton_Afunc(u, theta):
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    a = np.hstack((-theta*np.log(u), np.zeros_like(u0)))
    b = np.hstack((np.ones_like(u0), np.ones_like(u1), -np.ones_like(u0)))

    return np.exp(special.logsumexp(a, b=b, axis=1)).reshape(-1,1) #u0**(-theta) + u1**(-theta) - 1

def clayton_logAfunc(u, theta):
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    a = np.hstack((-theta*np.log(u), np.zeros_like(u0)))
    b = np.hstack((np.ones_like(u0), np.ones_like(u1), -np.ones_like(u0)))
    #print(a.shape)
    #print(b.shape)

    return special.logsumexp(a, b=b, axis=1).reshape(-1,1)
    
def clayton_Afunc_grad(u, theta):
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    return -((u0**(-theta))*np.log(u0) + (u1**(-theta))*np.log(u1))

def clayton_logA_grad(u, theta):
    A_func = clayton_Afunc(u, theta)
    return clayton_Afunc_grad(u, theta)/A_func

def clayton_lpdf(u, theta):
    """
    Clayton copula log-likelihood evaluation

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Clayton copula log-likelihood
    """
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    A_func = clayton_Afunc(u, theta)
    #print(np.log(A_func))
    #print(clayton_logAfunc(u, theta))
    lpdf = special.log1p(theta) - (1+theta)*(np.log(u0)+ np.log(u1)) - (2+1/theta)*clayton_logAfunc(u, theta) #np.log(A_func)
    return np.sum(lpdf)

def clayton_lpdf_grad(u, theta):
    """
    Clayton copula gradient w.r.t copula parameter theta

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Clayton copula lpdf gradient
    """
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)

    A_func = clayton_Afunc(u, theta)
    #logA = clayton_logAfunc(u, theta)
    dlogA = clayton_logA_grad(u, theta)
    lpdf_grad = 1/(1+theta) - (np.log(u0) + np.log(u1)) - (2 + 1/theta)*dlogA + clayton_logAfunc(u, theta)/np.square(theta)
    return lpdf_grad
    
def clayton_link(par):
    """
    :param par: current parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: link function of copula parameter
    """
    return np.log(par)
    
def clayton_inv_link(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: inverse link function of transformed copula parameter
    """
    return np.exp(tpar)
    
def clayton_inv_link_grad(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    return: gradient of inverse link function w.r.t transformed copula parameter
    """
    return np.exp(tpar)

def clayton_gradient(u: np.ndarray, predt: np.ndarray) -> np.ndarray:
    '''Compute the gradient of the nll'''
    #u = dtrain.get_label()
    tau = special.expit(predt)
    #rho = gauss_tau2par(tau)
    theta = clayton.clayton_tau2par(tau)
    return -clayton_lpdf_grad(u, theta)*(2*tau/(1-tau)) # explain # *clayton_inv_link_grad(predt)
        
def clayton_copula_nll_obj(u: np.ndarray, predt: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    '''Clayton copula negative log-likelihood objective'''
    grad = clayton_gradient(u, predt)
    hess = grad**2 #BHHH hessian substitute
    return grad, hess

def clayton_copula_nll_metric(u: np.ndarray, predt: np.ndarray) -> Tuple[str, float]:
    ''' Clayton copula negative log-likelihood metric'''
    #u = dtrain.get_label()
    tau = special.expit(predt)
    #rho = gauss_tau2par(tau)
    theta = clayton.clayton_tau2par(tau) #clayton_inv_link(predt)
    log_likelihood = clayton_lpdf(u, theta)
    return float(-log_likelihood)

In [8]:
def gumbel_lpdf(u, theta):
    """
    Gumbel copula log-likelihood evaluation

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Gumbel copula log-likelihood
    """
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    t = (-np.log(u0))**theta
    s = (-np.log(u1))**theta
    a = (t + s)**(1/theta)

    return np.sum(-a - np.log(u0*u1) + (-2+2/theta)*np.log(t+s) + (theta - 1)*(np.log(np.log(u0)*np.log(u1))) + np.log(1+(theta-1)/a))
    
def gumbel_lpdf_grad(u, theta):
    """
    Gumbel copula gradient w.r.t dependence parameter theta

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param theta: current dependence parameter estimates (n_obs, 1)
        
    :return: Gumbel copula lpdf gradient
    """
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    t = (-np.log(u0))**theta
    s = (-np.log(u1))**theta
    a = (t + s)**(1/theta)
    b = np.log(t+s)/np.square(theta)
    d = (t*np.log(-np.log(u0)) + s*np.log(-np.log(u1)))/(t+s)

    return -a*(-b + d/theta) + (-2*b + (-2+2/theta)*d) + np.log(np.log(u0)*np.log(u1)) + a/(a + (theta - 1))*(1/a + ((theta-1)/a)*(b - d/theta))
    
def gumbel_link(par):
    """
    :param par: current parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: link function of copula parameter
    """
    return np.log(par-1)
    
def gumbel_inv_link(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: inverse link function of transformed copula parameter
    """
    return np.exp(tpar)+1
    
def gumbel_inv_link_grad(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    return: gradient of inverse link function w.r.t transformed copula parameter
    """
    return np.exp(tpar)

def gumbel_gradient(u: np.ndarray, predt: np.ndarray) -> np.ndarray:
    '''Compute the gradient of the nll'''
    #u = dtrain.get_label()
    #tau = gauss_par2tau(copula_inv_link(predt))
    #rho = gauss_tau2par(tau)
    tau = special.expit(predt)
    #rho = gauss_tau2par(tau)
    theta = gumbel.gumbel_tau2par(tau)
    #theta = gumbel_inv_link(predt)
    return -gumbel_lpdf_grad(u, theta)*(2*tau/(1-tau)) #*gumbel_inv_link_grad(predt)
        
def gumbel_copula_nll_obj(u: np.ndarray, predt: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    '''Gumbel copula negative log-likelihood objective'''
    grad = gumbel_gradient(u, predt)
    hess = grad**2 #BHHH hessian substitute
    return grad, hess

def gumbel_copula_nll_metric(u: np.ndarray, predt: np.ndarray) -> Tuple[str, float]:
    ''' Gumbel copula negative log-likelihood metric'''
    #u = dtrain.get_label()
    tau = special.expit(predt)
    #rho = gauss_tau2par(tau)
    theta = gumbel.gumbel_tau2par(tau)
    #tau = gauss_par2tau(gaussian_inv_link(predt))
    #rho = gauss_tau2par(tau)
    #theta = gumbel_inv_link(predt)
    log_likelihood = gumbel_lpdf(u, theta)
    return float(-log_likelihood)

In [9]:
from sklearn.metrics import make_scorer
from xgboost import XGBRegressor

gauss_scorer = make_scorer(gaussian_copula_nll_metric, greater_is_better=False)

gauss_model = XGBRegressor(
    objective=gaussian_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

clayton_scorer = make_scorer(clayton_copula_nll_metric, greater_is_better=False)

clayton_model = XGBRegressor(
    objective=clayton_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

gumbel_scorer = make_scorer(gumbel_copula_nll_metric, greater_is_better=False)

gumbel_model = XGBRegressor(
    objective=gumbel_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

In [18]:
from sklearn.model_selection import train_test_split, GridSearchCV

param_grid = {
    'max_depth': [1, 2, 3, 5],
    'eta':[0.01, 0.1, 0.3],
    'lambda': [0.0, 0.1, 0.2]
    #'subsample': [0.6, 0.8, 1.0],
    #'colsample_bytree': [0.6, 0.8, 1.0]
}

grid_search_gauss=GridSearchCV(
    estimator=gauss_model,
    param_grid=param_grid,
    scoring=gauss_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_clayton=GridSearchCV(
    estimator=clayton_model,
    param_grid=param_grid,
    scoring=clayton_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_gumbel=GridSearchCV(
    estimator=gumbel_model,
    param_grid=param_grid,
    scoring=gumbel_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

In [19]:
y_gauss1 = sample_gaussian_copula(gauss.gauss_tau2par(tau1), 0)
x_train_gauss1, _, y_train_gauss1, _ = train_test_split(x, y_gauss1, test_size=1-ratio, random_state=42)

grid_search_gauss.fit(x_train_gauss1, y_train_gauss1)

print(f"Best parameters (gauss1): {grid_search_gauss.best_params_}")
print(f"Best score (gauss1): {grid_search_gauss.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gauss1): {'eta': 0.01, 'lambda': 0.1, 'max_depth': 1}
Best score (gauss1): 407.2523798354931


In [20]:
y_clayton1 = sample_clayton_copula(clayton.clayton_tau2par(tau1), 0)
x_train_clayton1, _, y_train_clayton1, _ = train_test_split(x, y_clayton1, test_size=1-ratio, random_state=42)

grid_search_clayton.fit(x_train_clayton1, y_train_clayton1)

print(f"Best parameters (clayton1): {grid_search_clayton.best_params_}")
print(f"Best score (clayton1): {grid_search_clayton.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


c:\Users\antho\master-thesis-project\xgb_env\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [552.16209648 552.37903281 545.58728642 443.01763444 552.15495421
 552.51065934 545.66434045 447.29621348 552.14926071 552.54825931
 545.45247396 448.89997137 557.10589912          nan          nan
          nan 557.14189591 409.7456637           nan          nan
 557.27027513          nan          nan          nan 537.95599535
          nan          nan          nan 538.28126024          nan
          nan          nan 538.55284382          nan          nan
          nan]
  warnings.warn(


Best parameters (clayton1): {'eta': 0.1, 'lambda': 0.2, 'max_depth': 1}
Best score (clayton1): 557.2702751307888


In [21]:
y_gumbel1 = sample_gumbel_copula(gumbel.gumbel_tau2par(tau1), 0)
x_train_gumbel1, _, y_train_gumbel1, _ = train_test_split(x, y_gumbel1, test_size=1-ratio, random_state=42)

grid_search_gumbel.fit(x_train_gumbel1, y_train_gumbel1)

print(f"Best parameters (gumbel1): {grid_search_gumbel.best_params_}")
print(f"Best score (gumbel1): {grid_search_gumbel.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


c:\Users\antho\master-thesis-project\xgb_env\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [487.05032154 488.8613968  481.13916093 425.97221599 487.04894797
 489.03225025 481.32489205 422.91846696 487.04740964 489.17836813
 481.42405969 424.8941565  516.10134224 463.42003984          nan
          nan 516.15527055 460.14482516          nan          nan
 516.20502946 463.46949571          nan          nan 508.18576985
          nan          nan          nan 508.29959802          nan
          nan          nan 508.51139447          nan          nan
          nan]
  warnings.warn(


Best parameters (gumbel1): {'eta': 0.1, 'lambda': 0.2, 'max_depth': 1}
Best score (gumbel1): 516.2050294645594


In [22]:
y_gauss2 = sample_gaussian_copula(gauss.gauss_tau2par(tau2), 0)
x_train_gauss2, _, y_train_gauss2, _ = train_test_split(x, y_gauss2, test_size=1-ratio, random_state=42)

grid_search_gauss.fit(x_train_gauss2, y_train_gauss2)

print(f"Best parameters (gauss2): {grid_search_gauss.best_params_}")
print(f"Best score (gauss2): {grid_search_gauss.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gauss2): {'eta': 0.01, 'lambda': 0.2, 'max_depth': 2}
Best score (gauss2): 498.1717241717324


In [23]:
y_clayton2 = sample_clayton_copula(clayton.clayton_tau2par(tau2), 0)
x_train_clayton2, _, y_train_clayton2, _ = train_test_split(x, y_clayton2, test_size=1-ratio, random_state=42)

grid_search_clayton.fit(x_train_clayton2, y_train_clayton2)

print(f"Best parameters (clayton2): {grid_search_clayton.best_params_}")
print(f"Best score (clayton2): {grid_search_clayton.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


c:\Users\antho\master-thesis-project\xgb_env\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [633.27576169 641.18963808 628.97788051          nan 633.27113743
 641.3427354  630.65467324          nan 633.27146877 641.33553988
 630.92122782          nan 639.01031914          nan          nan
          nan 638.74271164          nan          nan          nan
 638.85230387          nan          nan          nan 623.7708731
          nan          nan          nan 623.79880642          nan
          nan          nan 624.13261083          nan          nan
          nan]
  warnings.warn(


Best parameters (clayton2): {'eta': 0.01, 'lambda': 0.1, 'max_depth': 2}
Best score (clayton2): 641.342735397507


In [24]:
y_gumbel2 = sample_gumbel_copula(gumbel.gumbel_tau2par(tau2), 0)
x_train_gumbel2, _, y_train_gumbel2, _ = train_test_split(x, y_gumbel2, test_size=1-ratio, random_state=42)

grid_search_gumbel.fit(x_train_gumbel2, y_train_gumbel2)

print(f"Best parameters (gumbel2): {grid_search_gumbel.best_params_}")
print(f"Best score (gumbel2): {grid_search_gumbel.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


c:\Users\antho\master-thesis-project\xgb_env\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [562.19586579 571.73267219 570.29186788 507.62982698 562.194019
 571.71616512 570.41705419 514.16254266 562.19218321 571.70628154
 570.46934698 511.95757385 605.61972283 567.08141649          nan
          nan 605.6196706  565.61908697          nan          nan
 605.5704906  568.50866361          nan          nan 596.22544258
          nan          nan          nan 595.81073062          nan
          nan          nan 595.50433707          nan          nan
          nan]
  warnings.warn(


Best parameters (gumbel2): {'eta': 0.1, 'lambda': 0.0, 'max_depth': 1}
Best score (gumbel2): 605.6197228325257
